## Source (OLTP DATABASE)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

data = [
    (101, "Rahul", 500, "Pending", "2026-05-15 10:00:00"),
    (102, "Neha", 1200, "Delivered", "2026-05-15 10:05:00"),
    (103, "Aman", 750, "Shipped", "2026-05-15 10:10:00")
]

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("order_status", StringType(), True),
    StructField("updated_at", StringType(), True)
])

df = spark.createDataFrame(data, schema)

df = df.withColumn(
    "updated_at",
    to_timestamp(col("updated_at"))
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ekart_dt.default.orders_source")

#Water table

In [0]:
watermark_data = [
    ("orders_pipeline", "2026-05-15 10:10:00")
]

watermark_schema = [
    "pipeline_name",
    "last_loaded_timestamp"
]

watermark_df = spark.createDataFrame(
    watermark_data,
    watermark_schema
)

watermark_df = watermark_df.withColumn(
    "last_loaded_timestamp",
    to_timestamp(col("last_loaded_timestamp"))
)

watermark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ekart_dt.default.etl_watermark")

#Read Watermark Value

In [0]:
watermark = spark.sql("""

SELECT last_loaded_timestamp
FROM ekart_dt.default.etl_watermark
WHERE pipeline_name = 'orders_pipeline'

""").collect()[0][0]

print(watermark)

# Incremental Extraction

In [0]:
incremental_df = spark.sql(f"""

SELECT *
FROM ekart_dt.default.orders_source
WHERE updated_at > TIMESTAMP('{watermark}')

""")

incremental_df.show()

# Simulate New Incoming Data in source

In [0]:
new_data = [
    (104, "Priya", 900, "Pending", "2026-05-15 10:20:00"),
    (102, "Neha", 1200, "Returned", "2026-05-15 10:25:00"),
    (106,"Neha",12000,"Pending","2026-05-15 10:30:00")
]

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("order_status", StringType(), True),
    StructField("updated_at", StringType(), True)
])

new_df = spark.createDataFrame(new_data, schema)

new_df = new_df.withColumn(
    "updated_at",
    to_timestamp(col("updated_at"))
)

new_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("ekart_dt.default.orders_source")

# Read Incremental Data Again

In [0]:
incremental_df = spark.sql(f"""

SELECT *
FROM ekart_dt.default.orders_source
WHERE updated_at > TIMESTAMP('{watermark}')

""")

incremental_df.show()

#Create Target Table (First Time) OLAP(DATAWAREHOUSE)
)

In [0]:
source_df = spark.table("ekart_dt.default.orders_source")
source_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ekart_dt.default.orders_target")

#MERGE Incremental Data into Target

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "ekart_dt.default.orders_target")

target.alias("t").merge(
    incremental_df.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdate(set={
    "customer_name": "s.customer_name",
    "amount": "s.amount",
    "order_status": "s.order_status",
    "updated_at": "s.updated_at"
}).whenNotMatchedInsert(values={
    "order_id": "s.order_id",
    "customer_name": "s.customer_name",
    "amount": "s.amount",
    "order_status": "s.order_status",
    "updated_at": "s.updated_at"
}).execute()

In [0]:
%sql
select * from ekart_dt.default.orders_target

# Update Watermark Table (To get latest timestamp)

In [0]:
new_watermark = incremental_df.agg(
    max("updated_at")
).collect()[0][0]

print(new_watermark)

#Update Watermark

In [0]:
spark.sql(f"""

UPDATE ekart_dt.default.etl_watermark
SET last_loaded_timestamp = TIMESTAMP('{new_watermark}')
WHERE pipeline_name = 'orders_pipeline'

""")

In [0]:
%sql
select * from ekart_dt.default.orders_target

In [0]:
%sql
select * from ekart_dt.default.etl_watermark